<a href="https://colab.research.google.com/github/salmanmfa2/data-science-2026/blob/main/Pertemuan3_Salman_MFA_240401010356.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import pandas as pd
import numpy as np
import missingno as msno
import scipy.stats.mstats as winsorize
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

In [39]:
url = 'https://drive.google.com/uc?id=1qvz1quon6nG8ke8bDYh6HwPc75S8vKIJ'
df = pd.read_csv(url)
print(df.shape, df.columns.tolist())
print(df.isnull().sum())


(130, 7) ['id', 'luas_m2', 'harga_juta', 'kota', 'kamar', 'tahun_bangun', 'kondisi']
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


CEK DUPLICATED

In [40]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")

Number of duplicate rows: 0


In [41]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None


In [42]:
print(df.describe())

               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.450000e+02    2.000000   1991.250000
50%     65.500000   193.800000  6.550000e+02    4.000000   2002.000000
75%     97.750000   280.675000  9.550000e+02    5.000000   2011.750000
max    130.000000  9500.000000  1.000000e+08    6.000000   9999.000000


CEK Null

In [43]:
print (df.isnull().sum())

id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [44]:
pct = (df.isnull().sum() / len(df) * 100).round(2)

In [45]:
print(pct)

id               0.00
luas_m2         13.85
harga_juta      13.08
kota             0.00
kamar            7.69
tahun_bangun     0.00
kondisi          0.00
dtype: float64


CEK dan Hapus Duplicates

In [46]:
print(df.shape)
df.drop_duplicates(inplace=True)
print(df.shape)

(130, 7)
(130, 7)


Pembersihan Kolom "Kota" dan "Kondisi"

In [47]:
df['kondisi'].unique()
df['kondisi'] = df['kondisi'].str.strip().str.lower()
print (df['kondisi'].unique())
dict_cleaning_kondisi = {
    'bagus':'baik',
    'cukup':'sedang',
    'jelek' : 'perlu renovasi',
}
df['kondisi'] = df['kondisi'].replace(dict_cleaning_kondisi).str.title()
print (df['kondisi'].unique())

['baik' 'bagus' 'sedang' 'baik sekali' 'rusak' 'cukup' 'perlu renovasi'
 'jelek']
['Baik' 'Sedang' 'Baik Sekali' 'Rusak' 'Perlu Renovasi']


In [48]:
df['kota'].unique()

array(['jogja', 'Medan', 'Depok', 'YGY', 'Jakarta', 'jakarta',
       'Yogyakarta', 'Bandung', 'Surabaya', 'dpk', 'sby', 'Makassar',
       'mdn', 'medan', 'Semarang', 'semarang', 'yogyakarta', 'Jogja',
       'JAKARTA', 'Smg', 'DEPOK', 'Bdg', 'makassar', 'surabaya',
       'MAKASSAR', 'depok', 'bandung', 'Bandung ', 'SURABAYA', 'Mksr',
       ' Jakarta'], dtype=object)

In [49]:
dict_cleaning_kota = {
    'jogja': 'DIY Yogyakarta',
    'ygy': 'DIY Yogyakarta',
    'yogyakarta': 'DIY Yogyakarta',
    'mdn': 'Medan',
    'medan': 'Medan',
    'jakarta': 'Jakarta',
    'depok': 'Depok',
    'dpk': 'Depok',
    'bandung': 'Bandung',
    'bdg': 'Bandung',
    'surabaya': 'Surabaya',
    'sby': 'Surabaya',
    'semarang': 'Semarang',
    'smg': 'Semarang',
    'makassar': 'Makassar',
    'mksr': 'Makassar',
}

df["kota"] = (df["kota"].str.strip().str.lower()).replace(dict_cleaning_kota)
print(df.head())


   id  luas_m2  harga_juta            kota  kamar  tahun_bangun kondisi
0   1    297.0      1084.0  DIY Yogyakarta    2.0          2000    Baik
1   2    254.0       761.0           Medan    NaN          1995    Baik
2   3    249.7       895.0           Depok    NaN          1983    Baik
3   4     49.7       178.0  DIY Yogyakarta    5.0          2013    Baik
4   5    133.4       424.0           Medan    5.0          2004  Sedang


Imputasi Missing Value

In [50]:
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])


Cek Hasil Cleaning

In [51]:
print(df.head())
print(df.isnull().sum())

   id  luas_m2  harga_juta            kota  kamar  tahun_bangun kondisi
0   1    297.0      1084.0  DIY Yogyakarta    2.0          2000    Baik
1   2    254.0       761.0           Medan    1.0          1995    Baik
2   3    249.7       895.0           Depok    1.0          1983    Baik
3   4     49.7       178.0  DIY Yogyakarta    5.0          2013    Baik
4   5    133.4       424.0           Medan    5.0          2004  Sedang
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


Deteksi Outlier

In [52]:
def deteksi_outlier_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    return lower_bound, upper_bound, outliers

In [53]:
low , up , outs = deteksi_outlier_iqr(df['luas_m2'])
print (low)
print (up)
print (outs)

-145.225
512.9749999999999
103    9500.0
Name: luas_m2, dtype: float64


In [54]:
# Outlier detection for 'luas_m2'
lower_luas_m2, upper_luas_m2, outliers_luas_m2_values = deteksi_outlier_iqr(df['luas_m2'])
print(f"luas_m2 Lower Bound: {lower_luas_m2:.2f}")
print(f"luas_m2 Upper Bound: {upper_luas_m2:.2f}")

# Get the IDs of outlier rows for 'luas_m2'
outlier_luas_m2_ids = df[ (df['luas_m2'] < lower_luas_m2) | (df['luas_m2'] > upper_luas_m2) ]['id']
print(f"Outlier IDs for luas_m2: {outlier_luas_m2_ids.tolist()}")

# Outlier detection for 'harga_juta'
lower_harga_juta, upper_harga_juta, outliers_harga_juta_values = deteksi_outlier_iqr(df['harga_juta'])
print(f"\nharga_juta Lower Bound: {lower_harga_juta:.2f}")
print(f"harga_juta Upper Bound: {upper_harga_juta:.2f}")

# Get the IDs of outlier rows for 'harga_juta'
outlier_harga_juta_ids = df[ (df['harga_juta'] < lower_harga_juta) | (df['harga_juta'] > upper_harga_juta) ]['id']
print(f"Outlier IDs for harga_juta: {outlier_harga_juta_ids.tolist()}")

# Outlier detection for 'tahun_bangun'
lower_tahun_bangun, upper_tahun_bangun, outliers_tahun_bangun_values = deteksi_outlier_iqr(df['tahun_bangun'])
print(f"\ntahun_bangun Lower Bound: {lower_tahun_bangun:.2f}")
print(f"tahun_bangun Upper Bound: {upper_tahun_bangun:.2f}")

# Get the IDs of outlier rows for 'tahun_bangun'
outlier_tahun_bangun_ids = df[ (df['tahun_bangun'] < lower_tahun_bangun) | (df['tahun_bangun'] > upper_tahun_bangun) ]['id']
print(f"Outlier IDs for tahun_bangun: {outlier_tahun_bangun_ids.tolist()}")

luas_m2 Lower Bound: -145.22
luas_m2 Upper Bound: 512.97
Outlier IDs for luas_m2: [104]

harga_juta Lower Bound: -422.75
harga_juta Upper Bound: 1719.25
Outlier IDs for harga_juta: [27, 80, 102]

tahun_bangun Lower Bound: 1960.50
tahun_bangun Upper Bound: 2042.50
Outlier IDs for tahun_bangun: [73, 84, 103]


### Full Row Details for Outliers

In [55]:
print('--- Outliers in luas_m2 ---')
display(df[df['id'].isin(outlier_luas_m2_ids)])

print('\n--- Outliers in harga_juta ---')
display(df[df['id'].isin(outlier_harga_juta_ids)])

print('\n--- Outliers in tahun_bangun ---')
display(df[df['id'].isin(outlier_tahun_bangun_ids)])

--- Outliers in luas_m2 ---


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
103,104,9500.0,179.0,DIY Yogyakarta,4.0,1985,Baik



--- Outliers in harga_juta ---


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
26,27,113.8,-500.0,Bandung,6.0,1993,Baik Sekali
79,80,193.8,1882.0,Jakarta,2.0,1986,Sedang
101,102,221.6,99999999.0,Medan,4.0,2022,Sedang



--- Outliers in tahun_bangun ---


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
72,73,193.8,894.0,Bandung,6.0,1890,Perlu Renovasi
83,84,218.5,696.0,Bandung,2.0,2099,Sedang
102,103,178.5,655.0,Medan,3.0,9999,Sedang
